## Evaluate a saved fund timeseries check with several LLMs

Reads one folder under `output-data/timeseries-checks/<ISIN>/` (written by the check notebook) and asks the same six questions to OpenAI, Anthropic, Google, DeepSeek, Groq, and TypeSafe.

TypeSafe returns only the four scores on a 0–1 scale (via a five-level rubric). The generative models also return qualitative interpretations.

API keys are loaded from `.env` and are not printed.

In [1]:
import json
import sys
from pathlib import Path

from dotenv import find_dotenv, load_dotenv


def _ensure_helpers_importable() -> None:
    start = Path.cwd().resolve()
    for directory in [start, *start.parents]:
        if (directory / "helpers").is_dir():
            root = str(directory)
            if root not in sys.path:
                sys.path.insert(0, root)
            return
    raise FileNotFoundError(f"Could not find helpers/, starting from {start}")


_ensure_helpers_importable()
load_dotenv(find_dotenv(usecwd=True))

from helpers.timeseries_functions.fund_llm_dossier import (
    SYSTEM_PROMPT,
    build_dossier,
    build_user_prompt,
    dossier_json,
    timeseries_check_dir,
)
from helpers.timeseries_functions.fund_llm_providers import (
    DEFAULT_GENERATIVE_MODELS,
    run_all_providers,
    scores_comparison_table,
)

SELECTED_FUND = "US4613088016"

GENERATIVE_MODELS = DEFAULT_GENERATIVE_MODELS.copy()
# GENERATIVE_MODELS["openai"] = "gpt-5-mini"

In [2]:
check_dir = timeseries_check_dir(SELECTED_FUND)
dossier = build_dossier(check_dir)

print(f"Loaded check folder: {check_dir}")
print(f"ISIN: {dossier['fund_profile']['isin']}")
print(f"Regression: {dossier['regression']['equation']}")
print(f"Mismatches: {dossier['source_comparison_summary']['mismatches']}")

preview = {
    "parameters": dossier["parameters"],
    "fund_profile": dossier["fund_profile"],
    "source_comparison_summary": dossier["source_comparison_summary"],
    "regression_statistics": dossier["regression"]["statistics"],
}
print(json.dumps(preview, indent=2))

Loaded check folder: /Users/olivierdedecker/Documents/02 - Enfants/Alix/Mémoire Alix/data/code/output-data/timeseries-checks/US4613088016
ISIN: US4613088016
Regression: Lipper = -0.06% + 0.927 × S&P 500 TR
Mismatches: 2
{
  "parameters": {
    "master_timeseries": "Lipper",
    "comparison_timeseries": "Yahoo",
    "index_column": "S&P 500 TR",
    "master_column": "US4613088016 (Lipper)",
    "comparison_column": "US4613088016 (Yahoo)",
    "match_tolerance": 0.0001,
    "spike_return": 0.1,
    "reversal_band": 0.15,
    "scale_factor": 5,
    "scale_window": 12,
    "level_jump": 0.5,
    "index_gap": 0.05,
    "date_gap_days": 45
  },
  "fund_profile": {
    "isin": "US4613088016",
    "market_identifier": "CICEX",
    "fund_wrapper": "mutual fund",
    "instrument_type": "Open-End Fund",
    "ric": "LP40076048",
    "benchmark_name": "S&P 500 TR",
    "total_expense_ratio": 0.84,
    "fund_inception_date": null
  },
  "source_comparison_summary": {
    "master_timeseries": "Lippe

In [3]:
user_prompt = build_user_prompt(dossier)
print("System prompt length:", len(SYSTEM_PROMPT))
print("User prompt length:", len(user_prompt))
print("Dossier JSON length:", len(dossier_json(dossier)))

System prompt length: 2104
User prompt length: 11838
Dossier JSON length: 10568


In [4]:
results = run_all_providers(dossier, models=GENERATIVE_MODELS)

for result in results:
    if result.ok:
        print(f"{result.provider} ({result.model}): ok")
    else:
        print(f"{result.provider} ({result.model}): {result.error}")

openai (gpt-5.4): ok
anthropic (claude-sonnet-5-5): ok
google (gemini-3.8-flash): HTTP 503: {
  "error": {
    "code": 503,
    "message": "This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.",
    "status": "UNAVAILABLE"
  }
}

deepseek (deepseek-v4-pro): ok
groq (openai/gpt-oss-120b): HTTP 403: error code: 1010

typesafe (jev-1.14.0): ok


In [5]:
import pandas as pd

display(scores_comparison_table(results))

for result in results:
    if result.provider == "typesafe":
        if result.raw:
            display(pd.DataFrame(result.raw).T)
        continue
    if not result.ok:
        continue
    print(f"\n=== {result.provider} ({result.model}) ===")
    if result.score_basis:
        for key, text in result.score_basis.items():
            print(f"[{key}] {text}")
    print("\nTimeseries interpretation:")
    print(result.timeseries_interpretation)
    print("\nRegression interpretation:")
    print(result.regression_interpretation)

,openai\n(gpt-5.4),anthropic\n(claude-sonnet-5-5),google\n(gemini-3.8-flash),deepseek\n(deepseek-v4-pro),groq\n(openai/gpt-oss-120b),typesafe\n(jev-1.14.0)
Price series reliable,0.94,0.88,—,0.90,—,0.68 (conf 0.73)
Tracks S&P 500 TR,0.18,0.10,—,0.10,—,0.54 (conf 0.50)
Active management (1=active),0.86,0.87,—,0.80,—,0.45 (conf 0.20)
Regression quality,0.82,0.82,—,0.90,—,0.57 (conf 0.55)



=== openai (gpt-5.4) ===
[price_timeseries_reliable] The timeseries checks are clean in both sources: gaps none, abnormal spikes none, scale shifts none, and differences with the S&P 500 TR above 5% none. Lipper and Yahoo match in 200 of 202 months (1 perfect, 199 acceptable), with only 2 mismatches and the worst issue explained as a date mismatch that reproduces the Lipper return within tolerance (gap 0.000006).
[tracks_sp500_tr] The fund moves closely with the index (R² 0.966, correlation 0.983), but the fitted beta is only 0.927 and annualized tracking error is 2.88%, which is far from the 'extremely close to 1' beta and well under 1% tracking error typical of an index tracker. The largest monthly gaps versus the index are also material, such as -269 bp in Aug 2022 and -259 bp in Sep 2018.
[active_management] Return behaviour is much more consistent with active management than passive index tracking: beta is 0.927, annualized tracking error is 2.88%, and annualized excess return ve

,score_0_to_1,confidence,probabilities
price_timeseries_reliable,0.675,0.73,"{0: 0.0, 1: 0.04, 2: 0.22, 3: 0.72, 4: 0.02}"
tracks_sp500_tr,0.5375,0.5,"{0: 0.03, 1: 0.16, 2: 0.47, 3: 0.3, 4: 0.04}"
active_management,0.445,0.2,"{0: 0.18, 1: 0.22, 2: 0.31, 3: 0.2, 4: 0.09}"
regression_quality,0.575,0.55,"{0: 0.01, 1: 0.1, 2: 0.49, 3: 0.38, 4: 0.02}"
